#### Automatic Summary Evaluation

This project demonstrates two LLM-as-a-judge techniques to evaluate summaries generated by machine learning models. Summary evaluation is useful application of LLMs in production environments and has analogies in similar evaluations of GenAI outputs (or the automated evaluation of human outputs).

This notebook uses data from SummEval [1] which is a dataset of CNN and Daily Mail articles, machine generated summaries of those articles, and human evaluations of those summaries. A G-Eval technique as well as a question-answer generation technique are used to evaluate the summaries on consistency (faithfulness).

Typically, LLM-as-a-judge means a variant of G-Eval [2] which is technique that essentially defines a metric to the LLM, then asks the LLM to rate some piece of input text using that metric. G-Eval is a powerful tool with relatively high correlation with human evaluations and low token and computation costs. It is especially powerful when the metric otherwise difficult to score - How would you the define the "relevance" of a summary using a rules-based-nlp method? How do you get enough "relevance" examples to train an NLP model?

G-Eval was originally used to better measure Relevance, Consistency, Fluency, and Coherence. Relevance being the measure of how well the summary captures the key points of the article. Consistency being the measure of whether the summary reproduces all facts accurately and does not make up untrue information. Fluency being the measure of the quality of individual sentences (grammatically correct). Coherence being the measure of the quality of the summary as a whole.

In general, machine written summaries are highly Fluent and Coherent [3] - to the point that focus as can shift entirely to Relevance and Consistency. Relevance is not as "solved" as Fluency or Coherence, but LLMs often produce output with acceptable levels of relevance. The biggest struggle for LLMs is often consistency - coming from an incorrect representation of events in the source text. These incorrect representations are known as hallucinations and can be an incorrect description of events that contradict the source (known as intrinsic hallucinations), or the introduction of new information that cannot be verified by the source (extrinsic information).

While G-Eval is a state-of-the-art metric, most of this notebook is dedicated to a question-answer generation (QAG) framework. QAG-based evaluation has been shown to be more correlated with human evaluation than G-Eval for consistency and similar metrics [4]. Similar metrics referring to faithfulness [3] (also known as groundedness) and coverage (also known as completeness), which are discussed further in the "Scoring the Summaries" section. SummEval's human evaluation of consistency is aligned with faithfulness, and so this notebook only compares the evaluation to faithfulness. The coverage scores for the summaries are generated, but are not compared to a ground truth.

The idea behind QAG is to use the summaries to generate questions, then (in a fresh call to the LLM that does not include the summary) have the LLM answer those questions using the source text. This method can also be used to generate questions from the source text and to answer those questions using the summaries. The details of why to do one vs the other are also given in the "Scoring the Summaries" section.

QAG is generally better in evaluating factuality-based tasks than G-Eval as QAG scores are more correlated with human evaluations than G-Eval scores. QAG also has the additional benefit of being interpretable as the questions and answers used to generate the scores are directly accessible. Interpretability is extremely useful in production environments with specific use cases as it allows engineers to diagnose undesirable behavior and mitigate risk. If certain aspects of the evaluation are important (Ex: need to ensure all named entities are present in the summary, need to ensure unique words/phrases are represented in the summary, etc.), we can verify that relevant questions are being generated. With G-Eval, we cannot "see" into how the model is generating the results, so if the model is "ignoring" part of the prompt, it can be more difficult to diagnose and mitigate those risks.

There is a downside to QAG in that it has significantly higher costs compared to G-Eval. These higher costs are in terms of both time and resources (compute/monetary) due to nearly 2 orders of magnitude more input and 3 orders of magnitude more output tokens. Notably, the cost per summary to run G-Eval using GPT4o-Mini with Azure OpenAI Service was (TODO) ms and $(TODO). The cost per summary to run the QAG method with the optimal number of questions was (TODO) ms and $(TODO).

The implementation of G-Eval uses the prompt from [5]. The implementation of QAG is custom and features several novel ideas such as LLM-answerability and the use of information density to generate a variable number of questions per summary. TODO: This implementation achieves better correlation with human evaluation than current state-of-the-art methods, which is curious as there is not much (if any) existing literature in using an LLM with a QAG framework. This implementation does pull inspiration from [6] and [7], but does not directly use any prompts or code from those sources.

In [1]:
import os
import json
from typing import Literal
import time
import asyncio
import aiohttp
from aiohttp import ClientResponseError
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
from scipy.stats import pearsonr, spearmanr, kendalltau

# Dataset paths
SUMM_EVAL_DATA = "./data/summEval/summeval.json"

# Hyperparameters for Question Generations
#   How many questions per document?
MIN_QUESTIONS = 6
MAX_QUESTIONS = 60
NUM_ITERATIONS = 10 # How many subdivisions in the search space between MIN_QUESTIONS and MAX_QUESTIONS

# LLAMA Parameters
#   Normally model params would be in a config file, but for simplicity for a demo notebook I use global variables
URL_LLAMA = "http://localhost:11434/api/chat"
QUESTIONS_MAX_LLAMA = 300 # Max number of questions per call - the retry logic allows us to get more total questions
QUESTION_BUFFER_LLAMA = 3 # Number of 'extra' questions per call to mitigate the effect of rejected questions
NUM_PARALLEL_CONNS_LLAMA = 3

# Azure OpenAI Service (GPT) Parameters
#   The structured output API is only supported by Azure OpenAI for GPT4o and GPT4o-Mini
URL_GPT = "https://{endpoint}/openai/deployments/{deployment-id}/chat/completions?api-version=2024-10-21"
AZURE_OPENAI_KEY = os.environ.get("AZURE_OPENAI_KEY")
NUM_REQUEST_RETRIES = 10 # If we have hit our token quota, we wait some time then try again (exponential backoff)
MAX_WAIT_TIME = 60 # Max time in seconds to sleep each time we wait
QUESTIONS_MAX_GPT = 450 # Max number of questions per call - the retry logic allows us to get more total questions
QUESTION_BUFFER_GPT = 2 # Number of 'extra' questions per call to mitigate the effect of rejected questions
NUM_PARALLEL_CONNS_GPT = 1

# Randomness Increments and Maxes
# If the model fails to generate any new answerable questions, we don't want to send the exact same prompt to the model,
#   because by default we have the randomness set as low as possible and it would generate the same (unuseful) output.
# If we would make a repeat call to the model, we need to introduce some randomness. If the call still fails to produce
#   new questions, we continue to increase the randomness up to some max value.
# Note: The calls are seeded, so once we hit the max value, the any further call will result in identical output. But
#   most of the time this method seems to get the model to generate enough questions. It only fails when the number of
#   questions >75
# Note: Azure OpenAI Service does not support top_k
TEMP_INCREMENT = 0.05
TOP_P_INCREMENT = 0.1
TOP_K_INCREMENT = 10
MAX_TEMP = 0.8
MAX_TOP_P = 0.90
MAX_TOP_K = 40

# Note: We will make calls to the model until we have enough questions or increment up to the max randomness.
#   This means we can't get stuck in an infinite loop because there is always an exit condition, but in practice it can
#   have a relatively high number of calls in the worst case scenario. In a production environment, this would have to
#   be better addressed, but in a demonstration of a novel method it is acceptable.

#### Reading in the Dataset

This dataset is from SummEval and comprised of full text (source) articles from CNN and Daily Mail, summaries generated by various NLP models, and human evaluations of those summaries.

For this notebook, the only score we are interested in is the consistency score. Consistency can have multiple definitions in the literature, so I will be using faithfulness to avoid ambiguity.

A summary is 'faithful' if the information in the summary is consistent with information in the source, and we do not consider ‘factuality’ where valid external facts are acceptable [3]. Not considering factuality is interesting because it means a factually inaccurate piece of information in the source should be accurately represented in the summary. Ex: A faithful summary of a flat-earther article would have the same factual inaccuracies that are present in the source article.

In [2]:
# Data used for evaluation available from the G-Eval paper's GitHub repo
#   https://raw.githubusercontent.com/nlpyang/geval/refs/heads/main/data/summeval.json
summ_eval = json.load(open(SUMM_EVAL_DATA))
summ_eval = pd.DataFrame(summ_eval)
for metric in ["coherence", "consistency", "fluency", "relevance"]:
    summ_eval[metric] = summ_eval["scores"].apply(lambda x: x[metric])
summ_eval.drop("scores", axis=1, inplace=True)

# Sanity check that the number of summaries and articles are the same
summaries = summ_eval["system_output"]
articles = summ_eval["source"]
assert (len(articles) == len(summaries))

#### Define the Search Grid

Similar to the MQAGs paper [4], a search over a the number of questions to generate is performed. One search is performed over a fixed number of questions per summary, and another search is performed using a fixed information density.

A fixed number of questions assumes that long summaries and short summaries have the same amount of information, and the long summaries are simply more verbose. Even within the SummEval dataset, which is a relatively clean dataset, the mean length of a summary is ~345 characters, but the standard deviation is ~90 (and the lengths are roughly normally distributed). roughly 1/6th of summaries are shorter than ~255 characters, and roughly 1/6th are longer than ~435 characters. While some of the summaries are simply more verbose, reading the summaries does seem to indicate that longer summaries contain more information.

Intuitively, it would make sense that the longer summaries require more questions to accurately capture all information. Using information density rather than a fixed number of questions assumes that longer summaries have more information than shorter summaries. For production applications, such as summarizing phone transcripts, it is likely that a longer transcript and/or longer summary contain more information, and the evaluation method would need more questions to accurately score the summary.

TODO: To jump ahead, the analysis shows that using information density only marginally outperforms a fixed number of questions, and only in some cases. This seems to be due to the variety of models used to generate the summaries as some are much better than others. A significant amount of the difference in summary length can be attributed to the "writing style" of the different models rather than the information density. However, for an analogy to production data where all the summaries are written by the same model, a longer summary is likely to be the result of more information rather than a change in the "writing style".

In [3]:
# Define a search grid based on a fixed number of questions
num_questions_search_grid = np.linspace(MIN_QUESTIONS, MAX_QUESTIONS, NUM_ITERATIONS)

# Define the search grid using information density
#   Each summary will have a variable number of questions generated
# The avg number of questions generated using information density is approximately equal to the fixed number of questions
#   This allows a roughly apples-to-apples comparison since the total number of questions generated will also be equal
# Note: In practice, the average number of questions (and actual number of questions) is not exactly equal as the model
#   can fail to produce enough questions. However, the numbers are close enough that it is still a fair comparison
mean_summary_length = np.mean([len(summary) for summary in summaries])
info_density_search_grid = 1 / np.linspace(MIN_QUESTIONS / mean_summary_length, MAX_QUESTIONS / mean_summary_length, NUM_ITERATIONS)

#### Calling the models

This notebook supports 2 models: llama3.2-vision (11b), and GPT4o-Mini. The Llama model is called using a local OLLAMA server. GPT4o-mini is called using the Azure OpenAI Service. It would be easy to add support for other models, especially other Llama or GPT models using OLLAMA or the Azure OpenAI Service. I am limited to the llama3.2-vision setup by my personal computer's RAM (and the processing time), and I am limited to GPT4o-Mini by the monetary cost of running the 

Both use a similar REST API endpoint, but there are some small differences between the APIs. All differences are contained and managed by the call_llm and get_request_payload functions.

In [4]:
def get_model_type(model_name: Literal["llama3.2-vision", "llama3.1:8b", "llama3.2:3b", "gpt4o-mini"]):
    """ Returns llama if the model is a llama model and returns gpt if the model is a gpt model
    """
    if model_name in ["llama3.2-vision", "llama3.1:8b", "llama3.2:3b"]:
        return "llama"
    elif model_name in ["gpt4o-mini"]:
        return "gpt"

def get_model_question_params(model_name):
    """ Returns the max number of questions per call and the question buffer for the given model
    
    QUESTIONS_MAX_*: int. Each model can only handle outputting so many questions before it "goes off the rails". All
        but the largest Llama models tend to start endlessly generating questions (until they hit their output token
        limit) if QUESTIONS_MAX is too large. GPT can handle more questions per call, but based on preliminary manual
        evaluation, the questions later in the output tend to be slightly less relevant.
    QUESTION_BUFFER_*: int. Whenever we generate questions, some of the questions will be unanswerable based on the
        question text (the model wrote the question from the summary, but then can't correctly answer its own question
        when re-given the summary). To mitigate the effects of this, we generate slightly more questions than needed
        and if only a small number get rejected, we save outselves an additional call to the model
    """
    model_type = get_model_type(model_name)
    if model_type == "llama":
        return QUESTIONS_MAX_LLAMA, QUESTION_BUFFER_LLAMA
    else:
        return QUESTIONS_MAX_GPT, QUESTION_BUFFER_GPT

In [5]:
# When we iterate over the search spaces, and especially because there are 16 summaries per source article, there are
#   many questions that get asked multiple times. 
# To speedup the computation, I use a cache. This also helped with experimentation with as the retry logic and other
#   features, and also helped while debugging.

# Using the try-except in this manner is just so we can re-run the full notebook without accidentally overwritting the
#   cache. The cache is also pickled every so often during inference to save the cache long term.
# TODO: note on system fingerprint for GPT. Note on how if/when different computations of ollama are on GPU vs CPU it can produce different results as well
try:
    CACHE
except:
    if os.path.exists("llama_cache.pickle"):
        with open("llama_cache.pickle", "rb") as f:
            CACHE = pickle.load(f)
    else:
        CACHE = dict()

In [6]:
# Calling Model

# Headers for Azure OpenAI calls
GPT_4O_HEADERS = {
    "Content-Type": "application/json",
    "api-key": AZURE_OPENAI_KEY
}


async def call_llm(session, model_name, request_payload, returned_stopped_only=True):
    """ Calls the given model using the given request payload
    """
    global CACHE
    
    # Get the model type (llama or GPT)
    model_type = get_model_type(model_name)
    
    # Get a hashable version of the payload - we will be caching the results in a dict
    if model_type == "llama":
        hashable_payload = tuple(
            [model_name] +
            [tuple(msg.items()) for msg in request_payload["messages"]] + 
            [tuple(option) for option in request_payload["options"].items()]
        )
    else:
        hashable_payload = tuple(
            [model_name] +
            [tuple(msg.items()) for msg in request_payload["messages"]] +
            [request_payload[item] for item in ["temperature", "top_p"]]
        )

    # If this request has already been sent, use the cached version
    if hashable_payload in CACHE:
        return CACHE[hashable_payload]

    # Since the request is new, make a call to the LLM
    # Llama and Azure OpenAI have slightly different APIs, so we need to call them slightly differently
    if model_type == "llama":
        try:
            async with session.post(URL_LLAMA, json=request_payload) as resp:
                response = await resp.json()
                resp.raise_for_status()
            retval = response["message"]["content"]

            # Both models have a field in the response to denote if the request finished due to the model supplying an 
            #   <|endoftext|> token, or due to hitting the output token limit
            # If the model hits the output token limit, simply return None
            if returned_stopped_only and response["done_reason"] != "stop":
                retval = None

        except ClientResponseError as e:
            print("LLAMA ClientResponseError", e)
            retval = None
    else:
        for i in range(NUM_REQUEST_RETRIES):
            try:
                async with session.post(URL_GPT, headers=GPT_4O_HEADERS, json=request_payload) as resp:
                    response = await resp.json()
                    resp.raise_for_status()
                
                # TODO: comments
                if returned_stopped_only and response["choices"][0]["finish_reason"] != "stop":
                    # Don't save this to the cache since it may be a max_tokens issue
                    return None 
                else:
                    retval = response["choices"][0]["message"]["content"]

            except ClientResponseError as e:
                # 429 is for quote rate limit reached. If the request failed due to quota, wait then try again
                # If the request failed due to any other reason, raise an error
                if resp.status != 429:
                    print("GPT ClientResponseError", e)
                    print("Request:", request_payload)
                    print("Response:", json.dumps(response))
                    # Don't save this to the cache since it is likely a network issue
                    return None

                time.sleep(min(MAX_WAIT_TIME, np.power(2, i)))

    CACHE[hashable_payload] = retval
    return retval


def get_request_payload(model_name, prompt_type, system_prompt, user_message, num_questions=None, randomness_options=None):
    """ Creates the request payload for the given model
    """
    # Set some fields based on the inputs
    model_type = get_model_type(model_name)
    num_output_tokens = 10 + 50*num_questions if prompt_type == "questions" else 2
    seed = 314159265

    # If randomness options were not supplied, set them to minimum values
    if randomness_options is None:
        randomness_options = {
            "temperature": 0.0,
            "top_p": 0.0,
            "top_k": 1
        }

    # Create the portions of the request payload that are identical between Ollama and Azure OpenAI
    payload = {
        "messages": [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_message
            }
        ],
        "stream": False,
    }
    
    # Create the Ollama specific portions if this is a llama request
    if model_type == "llama":
        payload["model"] = model_name
        payload["options"]= {
            "seed": seed,
            "temperature": randomness_options["temperature"],
            "top_p": randomness_options["top_p"],
            "top_k": randomness_options["top_k"],
            "num_predict": num_output_tokens
        }
        if prompt_type == "questions":
            payload["format"] = "json"

    # Create the Azure OpenAI specific portions if this is a GPT request
    else:
        payload["seed"] = seed
        payload["temperature"] = randomness_options["temperature"]
        payload["top_p"] = randomness_options["top_p"]
        payload["max_tokens"] = num_output_tokens
        
        # Azure's API allows us to specify the output JSON schema, and boasts 100% adherance to that schema
        if prompt_type == "questions":
            prompt_type["response_format"] = {
                "type": "json_schema",
                "json_schema": {
                    "name": "output",
                    "schema": {
                        "type": "object",
                        "properties": {
                            "questions":{
                                "type": "array",
                                # "minItems": NUMBER_OF_QUESTIONS, # Not a supported feature yet
                                # "maxItems": NUMBER_OF_QUESTIONS, # Not a supported feature yet
                                "items":{
                                    "type": "string",
                                    "additionalProperties": False
                                }
                            }
                        },
                        "required": ["questions"],
                        "additionalProperties": False
                    },
                    "strict": True
                }
            }

    return payload

#### Scoring Metric

The correct answers to all generated questions is "yes" (discussed later in the Question - Answer Generation section). The QAG score for a given summary is the percentage of answers that are "yes". This is equivalent to the one-best method from the MQAGs paper.

Scoring using a distance metric such as Total Variation is not possible with Ollama as the logprobs are not yet available as part of the response API. The logprobs could be approximated with multiple model calls and non-zero temperature, but that would balloon the processing time for what is likely to be a small improvement in correlation. The MQAG authors noted a relatively small difference in correlation with human scores between the best and worst distance metric. The "best" distance metric also changed depending on the dataset.

Once all summaries are scored (either using QAG or G-Eval), the correlation between those scores and the human evaluations is measured. Correlation is measured using Pearson, Spearman, and Kendall-Tau correlation. Pearson correlation measure the linear relationship between the machine scores and human evaluations while Spearman and Kendall-Tau look more so at the ranking of the data and compare the monotonic relationship between the machine scores and human evaluations. Kendall-Tau is typically more robust when dealing with tied ranks and will be the preferred metric of this analysis.

TODO: Notably, since Spearman and Kendall-Tau consider the ranking of the data, both methods are able to use ties to their advantage. Since G-Eval only scores the summary on a scale of 1-5, in practice there are 500-way ties for the scores of 3, 4, and 5. QAG similarly has many ties, but once the number of questions is greater than ~15, the number of ties per "score" is <100. From this perspective, G-Eval has an "easier" time ranking the summaries since it doesn't need to score the summaries so much as get them in the correct bin. In fairness, this is discussed in the G-Eval paper [2] as to why using a weighted average of the probabilities of the 1-5 scores underperforms the one-best token score. In order to make a better apples-to-apples comparison, I also include a binned correlation score where the QAG results are binned into scores of 1-5.

In [7]:
# Scoring metrics are based on correlation

# Calculate the QAG scores
def qag_score(qnas):
    scores = []
    for qna in qnas:
        y_pred = qna["answers"]
        y_true = np.full(y_pred.shape, "yes")
        scores.append(accuracy_score(y_true, y_pred))
    return scores

# Calculate the correlation between the method and summEval
def summEval_correlations(method_scores, summEval_scores):
    pearson = pearsonr(method_scores, summEval_scores)[0]
    spearman = spearmanr(method_scores, summEval_scores)[0]
    kendall_tau = kendalltau(method_scores, summEval_scores)[0]
    return pearson, spearman, kendall_tau

#### G-Eval Baseline

Since we are not using the model of the original paper (GPT4) we need to set a new G-Eval baseline to compare to the QAG results. The Llama models we use are significantly worse than GPT4 on text-based benchmarks, and GPT4o-mini is slightly worse (but a fraction, ~1/200th, of the monetary cost).

I tried to keep the prompt representative of the original G-Eval paper, but the Llama models had a hard time with the exact consistency prompt from the G-Eval paper. Issues with the G-Eval prompt and the Llama model ranged from not outputting just a number (it would output "On a scale of 1-5 I'd rate it a 4") to outputting an absurd number of newline characters or spaces after the score and more.

TODO: try the original G-Eval prompt with llama3.2-vision. Try the experimental G-Eval prompt from Microsoft PromptFlow https://github.com/microsoft/promptflow/tree/main/examples/flows/evaluation/eval-summarization#meta-evaluation 

TODO: try few-shot prompting. CoT was shown to be less performant than direct answers

This comparison is also slightly unfair to G-Eval because QAG has One-Shot prompting and effectively built-in Chain-of-Thought (CoT) prompting. An implementation of G-Eval with One-Shot (it would likely have to be Few-Shot Prompting with 1 example for each score 1-5) and CoT would product lift relative to Zero-Shot G-Eval. However, the authors of G-Eval notes that CoT decreased performance, and it would seem unlikely that even a good Few-Shot prompting would improve G-Eval performance enough to match QAG.

In [10]:
def g_eval_faithfulness_prompt(summary, article):
    system_prompt = f""""You are a super intelligent artificial intelligence that scores a summary based on its consistency with the source article. Your score is a rating from 1 to 5 (1, 2, 3, 4, 5) with 1 being a bad score and 5 being a good score.
Consistency is defined as the factual alignment between the summary and the source article. A factually consistent summary contains only statements that are entailed by the source article. Summaries that contain inconsistent information should be penalized.

You will be given a summary and the source article. Your task is to answer with the rated score, and only the score, for that summary.
Your answers should *strictly* be a number from 1 to 5. Do *not* output anything except a single number between 1 and 5.
Do *not* output additional information, comments, or context. Do *not* answer with any spaces, whitespace, newline characters or any other formatting.
"""

    user_message = f"""Source Article:
{article}

Summary:
{summary}
"""
    return system_prompt, user_message


async def g_eval_faithfulness(session, model_name, summary, article):
    system_prompt, user_message = g_eval_faithfulness_prompt(summary, article)
    response = await call_llm(
        session,
        model_name,
        get_request_payload(
            model_name,
            "geval",
            system_prompt,
            user_message
        ),
        returned_stopped_only=False
    )

    try:
        return int(response[0])
    except:
        print("response:", response)
        return None


async def g_eval_faithfulnesses(model_name):
    model_type = get_model_type(model_name)
    num_conns = NUM_PARALLEL_CONNS_LLAMA if model_type == "llama" else NUM_PARALLEL_CONNS_GPT

    connector = aiohttp.TCPConnector(limit=num_conns)
    timeout = aiohttp.ClientTimeout(total=3600)
    async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
        geval_tasks = [
            g_eval_faithfulness(session, model_name, summary, article)
            for summary, article in zip(summaries, articles)
        ]
        geval_results = await asyncio.gather(*geval_tasks)
    geval_correlations = summEval_correlations(geval_results, summ_eval["consistency"][:len(geval_results)])
    return geval_correlations


summaries = summ_eval["system_output"]
articles = summ_eval["source"]
g_eval_correlations ={
    "llama3.2:3b": await g_eval_faithfulnesses("llama3.2:3b"),
    "llama3.1:8b": await g_eval_faithfulnesses("llama3.1:8b"),
    "llama3.2-vision": await g_eval_faithfulnesses("llama3.2-vision"),
    # "gpt40-mini": await g_eval_faithfulnesses("gpt4o-mini"),
}
print(g_eval_correlations)

{'llama3.2:3b': (0.16941670431295536, 0.1830137739201798, 0.17531055839285614), 'llama3.1:8b': (0.5888296400923548, 0.4861699948990272, 0.45901144665677), 'llama3.2-vision': (0.6025404169925976, 0.45706823311205413, 0.42688800107452607)}


#### Prompt Engineering

TODO:
* one shot prompting
* system vs user message
* json structured output (both openAI GPT and LLAMA support it) - good way to put guardrails on model output
* instructions
* "you are a super artificial intelligence", "perfectly compliant JSON", headings ("-- Instructions --")
* tokenizer friendly spacing "-- Instructions --" instead of "--Instructions--"

In [11]:
# Here we define some examples for one-shot prompting
#   The summary and article examples are unrelated, but are both from the SummEval dataset
#   Neither the summary nor article are in the actual dataset used for evaluation

EXAMPLE_SUMMARY = "buckingham palace guard slipped on manhole cover in front of hundreds of horrified tourists. the queen's guard was left red-faced after he slipped on a manhole cover. he lost his footing and dropped his rifle. the incident was caught on camera. the guard is thought to have slipped because of metal shutters nailed to the soles of his boots."
EXAMPLE_SUMMARY_QUESTIONS = [
   "Did the buckingham palace guard slip on a manhole cover?",
   "Did the buckingham palace guard in front of tourists?",
   "Was the queen's guard left red-faced?",
   "Did the guard drop his rifle?",
   "Was the incident caught on camera?",
   "Is the guard thought to have slipped due to the metal shutters nailed to the soles of his boots?",
]

EXAMPLE_ARTICLE = """United Nations (CNN) -- The U.N. Security Council approved a resolution Monday to send 4,200 peacekeepers to Abyei, Sudan, as part of a recent agreement between Sudan and Southern Sudan. The resolution will establish, for six months, the United Nations Interim Security Force for Abyei (UNISFA), comprising "a maximum of 4,200 military personnel, 50 police personnel, and appropriate civilian support," the resolution states. It passed the council unanimously, 15-0. In a statement released by the State Department, Secretary Hiliary Clinton commended the swift passage of the resolution. "Abyei has been a source of regional tension for many years," the statement said. "We urge the parties to reach an immediate cease-fire and to provide aid workers with the unfettered access required to deliver humanitarian assistance to innocent civilians affected by the conflict." A week ago, the Sudanese government and the Sudan People's Liberation Movement signed an agreement to allow peacekeepers in Abyei, aimed at ending strife that has ravaged much of the country.  The two sides agreed in principle on the need for a third party to monitor the ill-defined border between north and south before the scheduled July 9 independence for the south. The U.N. peacekeepers will "monitor and verify the redeployment of any Sudan Armed Forces, Sudan People's Liberation Army or its successor" from the Abyei area, among other tasks, the Security Council resolution states."""
EXAMPLE_ARTICLE_QUESTIONS = [
   "Did the U.N. Security Council approve a resolution to send 4,200 peacekeepers to Abyei, Sudan?",
   "Was there a recent agreement between Sudan and Southern Sudan?",
   "Does the resolution establish the UNISFA?",
   "Did the resolution pass unanimously?",
   "Did Hiliary Clinton commend the swift passage of the resolution?",
   "Has Abyei been a source of regional tension for many years?",
   "Were Sudan and Southern Sudan urged to reach an immediate cease-fire?"
   "Were Sudan and Southern Sudan urged to provide aid workers with unfettered access to deliver humanitarian assistance?"
   "Did the Sudanese government sign an agreement a week ago?",
   "Did the Sudanese government allow peacekeepers in Abyei?",
   "Did the two parties agree in principal for a third party to monitor the ill-defined border?",
   "Is South Sudan scheduled for independence on July 9?",
   "Will U.N. peacekeepers monitor and verify the redeployment of Sudan Armed Forces, Sudan People's Liberation Army, and its successor?",
]

In [12]:
# Define the prompts for question generation

# First, we need to define some strings to insert into our prompts
question_json_schema_as_str = """{"questions": [< question 1 >, < question 2 >, ...]}"""

def get_base_question_system_prompt(num_questions, example_text, example_questions):
    example_questions_as_json_str = get_example_questions_as_json_str(example_questions)

    # We want to include a caveat with the example if the number of questions from the example differs from the number of questions we want to generate
    if len(example_questions) == num_questions:
        example_caveat = ""
    else:
        example_caveat = f"""There are {"more" if len(example_questions) > num_questions else "less"} than {num_questions} questions in the example, but the output must still have exactly {num_questions} questions. """

    base = """You are a super intelligent artificial intelligence that generates a perfectly compliant JSON with *exactly* {num_questions} close-ended questions based on text given from the user. You generate a diverse set of questions that can be answered with a 'yes' or a 'no', but the correct answer is always 'yes'. The schema for the JSON is {question_json_schema_as_str} where < question 1 > is the first question as a string, < question 2 > is the second question as a string, etc.
{retry_text}Included is an example text and example question JSON output. {example_caveat}Do *not* make questions based on the example.
The questions should test that a reader had full comprehension of the text.
The questions should be standalone - do not assume that previous questions are seen. This means do not use pronouns or any other similar implicit references to previous questions.
Do *not* output anything except the JSON with the correct schema. Do *not* output additional information, comments, or context. Do not include any newline characters. Keep the questions concise and relevant to the text.
Do *not* make up any information. Do *not* make any assumptions or inferences about the text

-- Instructions --
1) Think of a *strictly* close-ended yes-no question based on the text
2) Phrase the question so that the correct answer to the question must be 'yes'
3) {Instruction3}
4) Make sure the question is related to the given text and is NOT related to the example text
5) The question should be based *only* on the given text, and should not use any external information. The question *must* be able to be answered 'yes' using *only* the given text.
6) Add the question, and only the question to the JSON
7) Repeat steps 1 - 6 until *exactly* {num_questions} close-ended questions have been generated
8) Supply the final JSON and only the JSON without any additional information, comments or context

--- Example Text ---
{example_text}

--- Example Output ---
{example_questions_as_json_str}

--- End of Example ---"""
    base = base.replace("{question_json_schema_as_str}", question_json_schema_as_str)
    base = base.replace("{num_questions}", str(num_questions))
    base = base.replace("{example_text}", example_text)
    base = base.replace("{example_questions_as_json_str}", example_questions_as_json_str)
    base = base.replace("{example_caveat}", example_caveat)
    return base


def get_example_questions_as_json_str(example_questions):
    opening = '{"questions": ["'
    list_of_questions = '", "'.join(example_questions)
    closing = '"]}'
    return opening + list_of_questions + closing


def question_generation_system_prompt(num_questions, example_text, example_questions):
    system_prompt = get_base_question_system_prompt(num_questions, example_text, example_questions)
    system_prompt = system_prompt.replace("{retry_text}", "")
    system_prompt = system_prompt.replace("{Instruction3}", "Make sure this question is different from all previous questions")
    return system_prompt


def question_generation_retry_system_prompt(num_rety_questions, existing_questions, example_text, example_questions):
    system_prompt = get_base_question_system_prompt(num_rety_questions, example_text, example_questions)
    system_prompt = system_prompt.replace("{retry_text}", "Additionally included is a set of existing questions. Your questions must be different from the existing questions.\n")
    system_prompt = system_prompt.replace("{Instruction3}", "Make sure this question is different from the existing questions and all previous questions you have given")

    existing_questions_as_str = "\n".join(existing_questions)
    system_prompt +=f"""\n
--- Exising Questions ---
{existing_questions_as_str}

--- End of Existing Questions ---
"""
    return system_prompt


def question_generation_user_message(text):
    return f"""--- Given Text---
{text}

--- End Given Text---

--- Question JSON ---
"""


def question_generation_retry_user_message(text):
    return f"""--- Given Text---
{text}

--- End Given Text---

--- Question JSON ---
"""

In [13]:
# Define the prompts for question answering
def answer_generation_system_prompt():
    return f"""Based on the given text and close-ended question, answer that question with a 'yes' or 'no'. 
Answers should *strictly* be 'yes' or 'no'. Do *not* provide any other answer.
If the question is not close-ended and cannot be answered by a 'yes' or 'no', respond with 'no' for your answer
If the given text does not contain the information relevant to answering the question, respond with 'no' for your answer
If the answer is unknown based on the given text, then make the answer 'no', respond with 'no' for your answer
If the answer is ambiguous based on the given text, then make the answer 'no', respond with 'no' for your answer
Do *not* output anything except the string 'yes' or 'no'. Do *not* output additional information, comments, or context. Do *not* use newline characters.
"""

def answer_generation_user_message(text, question):
    return f"""--- Given Text---
{text}

-- Question --
{question}
"""

#### Question - Answer Generation

For question generation, we generate a set of close-ended, answerable yes/no questions from each summary
* For faithfulness, questions are generated using the summary and answered using the article
* The idea to use close-ended questions comes from MQAG, and the idea to use the simple yes/no answers only comes from [ref]
* The prompt specifies that questions be written so that the answer is always "yes". This is intended so that each question extracts 1 piece of information from the text.
    * Allowing "no" answers would allow for non-information extraction questions - In the single-shot example from the article about Sudan/South Sudan the model could ask "Did Sudan have a fireworks show for South Sudan?" to which the answer is no, but asking those kinds of questions is not useful.

* TODO: write up on how the goal is for each question to pertain to 1 piece of information. Then our metrics become "What percentage of the summary is based on the article?" and "what percentage of information in the article is covered by the summary?"
    * We definitely ask more questions than information. This is because it is a Monte-Carlo method. If there is 6 pieces of information and we ask 6 questions, it is easy to ask 2 questions about 1 piece of information and 0 of another. If we ask 24 questions, each piece of information is likely to be covered several times, and the distribution will be more uniform.

The prompt specifies that all questions are correctly answered with "yes", and we verify this by passing the question-text to the answer generation
* Any questions that are not answered with "yes" (including invalid answers) are removed
* The authors of MQAG call this "answerability"
* Ensuring all answers based on the summary are "yes" using the model is a form of "consistency"
    * This is similar to Cycle-Consistency of cycle GANs and is used elsewhere in NLP such as translation where beginning with an English sentence, then translating to French and translating that French sentence back to English should yield the original sentence.

More on consistency
* When we generate a question based on the summary and answer it based on the article, the model responds with yes/no
* We assume that the question is well written, and the correct answer is yes based on the summary. We also assume the answer from the model is correct based on the article
* These assumptions can break for several reasons, not limited to (forgive me for anthropomorphizing the model for brievity):
    * The model incorrectly interpreted the summary and wrote a question with an answer should be "no" based on the summary
    * The model incorrectly interpreted the article and gave the wrong answer
    * The answer prompt is "bad" and even though the model interpreted the article text correctly, it gave the wrong answer
    * The model incorrectly interpreted the question and gave the wrong answer
* By asking the model to answer the question based on the summary we can mitigate 3/4 of those issues
    * If the model incorrectly interprets the answer-text, we can't fix that
* ~13% of questions generated by the Llama model are rejected for being unanswerable (inconsistent)
    * An ablative study showed that consistency improved correlation with human evaluations from X to Y

TODO:
* Why to generate questions based on the summary and answers based on the article? Why not the reverse?

Details of Implementation
* There are some ugly nitty gritty details of the implementation
* TODO:
    * Max questions per call - Llama goes off the rails. GPT produces too few or too many questions
    * Retry logic. For 1 because if we want 50 questions but can only ask 15 in one call, we need at least 4 calls. For 2 because questions will be rejected due to being unanswerable. For 3, the models sometimes don't generate the output in the correct format so the output has to be thrown out
    * We start with 0 randomness - 0 temperature, top_p=0, top_k=1. We generate some questions, then check that they are answerable. If they are, when we do the retry prompt the previous questions go in the prompt. Every time we generate new questions, the list of previous questions grows. If we ever have a model output with 0 answerable questions (either all were unanswerable or the output was not able to be parsed), then we bump up the randomness a little. If we didn't, we would send the exact same prompt back to the model since the list of previous questions didn't change. If the randomness successfully generates a new question, we reset the randomness back to 0 and continue with the retry prompt as normal. If the small amount of randomness also leads to 0 answerable questions, we can bump up the randomness a little more. We can continue with more randomness until either a new question is successfully generated, or we hit some max randomness. If the max randomness still doesn't work, we give up. We could try changing the seed for the randomness until we get an answerable question, but there's no garantee that would work, and the amount of randomness needed will vary based on many factors. 

In [ ]:
# Define a function to generate questions/answers, and helper functions for questions/answers 
async def generate_answers(session, model_name, text, questions):
    """ Generate a list of answers to the given questions using an llm

    Returns a list of answers to the given list of questions based on the given text.
    Each answer is None or a String of "yes" or "no"
    LLM called using answer_generation_system_prompt for the system prompt and answer_generation_user_message for the
        user message

    Parameters:
    session - async aiohttp.ClientSession
    text - text to use to answer the list of questions
    questions - list of close-ended questions. Each question is a string and can be answered with 'yes' or 'no'
    """
    # Make all the answer calls in parallel
    answer_generation_tasks = [
        call_llm(
            session,
            model_name,
            get_request_payload(
                model_name,
                "answer",
                answer_generation_system_prompt(),
                answer_generation_user_message(text, question), 
            )
        )
        for question in questions
    ]
    raw_answers = await asyncio.gather(*answer_generation_tasks)

    # Lower case the answers
    lower_answers = [None if raw_answer is None else raw_answer.lower() for raw_answer in raw_answers]

    # Filter the answers to only be yes or no
    answers = [answer if answer in ["yes", "no"] else None for answer in lower_answers]
    return answers


async def generate_consistent_questions(session, model_name, text, num_questions, example_text, example_questions, previous_questions=[], randomness_options=None):
    """ Generate a list of close-ended questions that can be correctly answered with "yes" from the text using an llm

    Returns a list of close-ended questions (questions as strings) based on the given text where all answers are "yes".
    The model will be instructed to generate num_questions questions, but the actual output may have more or less.
    The output will likely be less since inconsistent questions are removed. But it is possible for the output to be
        more than the requested number, and for few or none to be inconsistent.
    Each question is "consistent" - generate_answers will return a list of all "yes" answers based on the input text.
    LLM called using question_generation_system_prompt for the system prompt and question_generation_user_message
        for the user message.

    Parameters:
    session - async aiohttp.ClientSession
    text - text to base the questions on
    num_questions - number of questions to tell the model to generate
    example_text - example text to use in the LLM's system prompt
    example_questions - example questions based on example_text to use in the LLM's system prompt
    previous_questions - Optional (default []]). If there are previous questions, use the retry prompt
    """
    # If there are no previous questions, use the initial question generation prompt
    if not previous_questions:
        questions_as_json_str = await call_llm(
            session,
            model_name,
            get_request_payload(
                model_name,
                "questions",
                question_generation_system_prompt(num_questions, example_text, example_questions),
                question_generation_user_message(text),
                num_questions,
                randomness_options
            ),
        )
    
    # If there are previous questions, use the retry system prompt
    else:
        questions_as_json_str = await call_llm(
            session,
            model_name,
            get_request_payload(
                model_name,
                "questions",
                question_generation_retry_system_prompt(num_questions, previous_questions, example_text, example_questions),
                question_generation_retry_user_message(text),
                num_questions,
                randomness_options
            )
        )

    # Get the list of questions from the string of the JSON object
    try:
        # If the response is None, that means the llm_call_func function flagged it for some reason - could be it
        #   terminated due to length rather than end of text, or a similar reason. See llm_call_func for details
        if questions_as_json_str is not None:
            questions_raw = json.loads(questions_as_json_str)["questions"]

        # If there was a problem, the initial call effectively returned no questions, and we will hope the retry logic
        #   does better
        else:
            questions_raw = []

        questions = []
        for question in questions_raw:
            if type(question) is str:
                questions.append(question)

    # If there was a problem, with the json string or it did not contain "questions", set the questions to a blank list
    except (json.decoder.JSONDecodeError, KeyError) as e:
        questions = []

    # First is a consistency check - the model should answer yes to all the questions when re-given the text
    #   This is a similar to the consistency loss used in Cycle-GAN (and other domains like language translation)
    consistency_answers = await generate_answers(session, model_name, text, questions)

    # Only keep the questions that are "consistent" as per the definition above
    #   Note: This also throws out answers of None (when the answer was invalid)
    questions_consistent = [
        question
        for question, answer in zip(questions, consistency_answers)
        if answer == "yes"
    ]

    return questions_consistent


async def generate_qnas(session, model_name, question_text, answer_text, chars_per_q, example_text, example_questions):
    """Generate a list of questions from the question_text and answers to those questions using answer_text

    Returns a dataframe with a column for questions a column for answers. 
    
    Each question is a string of a close-ended yes/no question based on question_text, and the answer to the question is "yes"
    The number of questions is determined by the length of question_text and chars_per_q (characters per question)
    Each question is "consistent" - generate_answers will return a list of all "yes" answers if given question_text
    
    Each answer is associated with a question and will be 

    Parameters:
    session - async aiohttp.ClientSession
    text - text to base the questions on
    chars_per_q - characters per question. Assumed information density of the text
    example_text - example text to use in the LLM's system prompt
    example_questions - example questions based on example_text to use in the LLM's system prompt
    """
    # Get the model specific question parameters
    questions_max, question_buffer = get_model_question_params(model_name)

    # Calculate the number of questions based on the given information density of the text
    num_questions_output = max(1, np.round(len(question_text) / chars_per_q, 0).astype(int).item())

    # Perform up to QAG_ATTEMPTS attempts to generate enough consistent questions and the answers to those questions
    questions = []
    answers = []
    prev_num_questions_i = 0
    prev_num_questions = 0
    ques_randomness = {
        "temperature": 0.0,
        "top_p": 0.0,
        "top_k": 1
    }
    max_randomness = False
    while (len(questions) < num_questions_output) and not max_randomness:
        # We need a total of num_questions_output generated. Usually, generate_consistent_questions produces less than
        #   this because some questions are inconsistent (but also the model is finicky and just give less questions)
        # To combat this, we tell the model to generate slightly more questions than we need
        #   If there still aren't enough questions, the for loop will repeat until we get enough or run out of attempts
        num_questions_i = num_questions_output + question_buffer

        # If nothing has changed since the last iteration, introduce some randomness
        #   This is so if the model failed to produce an output, we don't generate the same faild output repeatedly
        if prev_num_questions == len(questions) and prev_num_questions_i == num_questions_i:
            # The max values for temp, top_p, and top_k are the default Ollama values
            ques_randomness["temperature"] = min(MAX_TEMP, ques_randomness["temperature"] + TEMP_INCREMENT)
            ques_randomness["top_p"] = min(MAX_TOP_P, ques_randomness["top_p"] + TOP_P_INCREMENT)
            ques_randomness["top_k"] = min(MAX_TOP_K, ques_randomness["top_k"] + TOP_K_INCREMENT)

            # If it's reached the max amount of randomness, set the max_randomness flag
            if ques_randomness == {"temperature": MAX_TEMP, "top_p": MAX_TOP_P, "top_k": MAX_TOP_K}:
                max_randomness = True

        # Generate new questions
        questions_new = await generate_consistent_questions(
            session,
            model_name,
            question_text,
            num_questions_i,
            example_text,
            example_questions,
            questions,
            ques_randomness
        )

        # From those questions, generate new answers
        answers_new = await generate_answers(session, model_name, answer_text, questions_new)

        prev_num_questions_i = num_questions_i
        prev_num_questions = len(questions)

        # Append the new questions/answers with valid answers to the running lists
        for question, answer in zip(questions_new, answers_new):
            if answer in ["yes", "no"]:
                questions.append(question)
                answers.append(answer)

        # If the number of questions changed since the last iteration, reset the randomness back to 0
        if prev_num_questions != len(questions):
            ques_randomness = {
                "temperature": 0.0,
                "top_p": 0.0,
                "top_k": 1
            }
            max_randomness = False

        if len(questions) > 0: # TODO: delete me
            break # TODO: delete me

    # Truncate down the desired number of questions and check if we have enough. If we do, stop iterating
    questions = questions[:num_questions_output]
    answers = answers[:num_questions_output]

    if len(questions) < num_questions_output:
        print("Not enough questions", len(questions), num_questions_output)

    print(questions)
    return pd.DataFrame({"questions": questions, "answers": answers})


#### Async Processing

We have 1600 pairs of summaries and articles to churn through. The model calls take a long time, so they are performed asynchronously. Processing time is effectively directly proportional to the GPU bandwidth (or Azure OpenAI token rate limit) available.

The processing time is particularly inflated because we are investigating the effect of the number of questions, as well as information density vs fixed number of questions. This is effectively "training" on the data, and test time would be much less since we would know the information density or fixed number of questions.

In [15]:
async def qag_eval(model_name):
    # To compare to the fixed number of questions, we can calculate the mean number of questions
    #   The distribution of number of questions per summary is not very skewed, so the mean is roughly equal to the median
    # Using the mean allows us to compare to the fixed number of questions because if the fixed number equals the mean the
    #   the total number of questions is the same for both trials (and the computation should be roughly the same)
    information_density_mean_num_questions = []

    # No marginal increase in performance past 3 for Llama (on my machine)
    model_type = get_model_type(model_name)
    num_conns = NUM_PARALLEL_CONNS_LLAMA if model_type == "llama" else NUM_PARALLEL_CONNS_GPT
    connector = aiohttp.TCPConnector(limit=num_conns)

    # Set the timeout to 24 hours (gross overestimate)
    timeout = aiohttp.ClientTimeout(total=86400)

    # Generate questions and answers to score faithfulness (aka groundedness)
    #   How much of the information in the summary is faithful to (aka grounded in) the article?
    async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
        # Number of questions is fixed for each summary
        faithfulness_fixed_number_correlation_sets = []
        for num_questions in num_questions_search_grid:
            faithfulness_tasks = [
                generate_qnas(
                    session,
                    model_name,
                    summary,
                    article,
                    len(summary) / num_questions, # we need to calc info density per summary based on the fixed number
                    EXAMPLE_SUMMARY,
                    EXAMPLE_SUMMARY_QUESTIONS,
                )
                for summary, article in zip(summaries, articles)
            ]
            print("Generating Faithfulness QnAs. Num Question:", num_questions)
            faithfulness_qna_sets = await asyncio.gather(*faithfulness_tasks)

            faithfulness_scores = qag_score(faithfulness_qna_sets)
            correlations = summEval_correlations(faithfulness_scores, summ_eval["consistency"][:len(faithfulness_scores)])
            faithfulness_fixed_number_correlation_sets.append(correlations)
            print("\t", np.mean([len(qna["answers"]) for qna in faithfulness_qna_sets]))
            print("\t", correlations)

            with open("llama_cache.pickle", "wb") as f:
                pickle.dump(CACHE, f, pickle.HIGHEST_PROTOCOL)

        # Number of questions depends on assumed information density, and number of questions per summary is variable
        faithfulness_info_density_correlation_sets = []
        for chars_per_q in reversed(info_density_search_grid):
            faithfulness_tasks = [
                generate_qnas(
                    session,
                    model_name,
                    summary,
                    article,
                    chars_per_q,
                    EXAMPLE_SUMMARY,
                    EXAMPLE_SUMMARY_QUESTIONS,
                )
                for summary, article in zip(summaries, articles)
            ]
            print("Generating Faithfulness QnAs. Characters per Question:", chars_per_q)
            faithfulness_qna_sets = await asyncio.gather(*faithfulness_tasks)

            faithfulness_scores = qag_score(faithfulness_qna_sets)
            correlations = summEval_correlations(faithfulness_scores, summ_eval["consistency"][:len(faithfulness_scores)])
            faithfulness_info_density_correlation_sets.append(correlations)

            mean_num_questions = np.mean([len(qna["answers"]) for qna in faithfulness_qna_sets])
            information_density_mean_num_questions.append(mean_num_questions)
            print("\tMean:", mean_num_questions)
            print("\t", correlations)
            print()

            with open("llama_cache.pickle", "wb") as f:
                pickle.dump(CACHE, f, pickle.HIGHEST_PROTOCOL)

        # # Generate questions and answers to score coverage (aka completeness)
        # #   How much of the information in the article is coverd by the summary?
        # coverage_tasks = [
        #     generate_qnas(
        #         session,
        #         article,
        #         summary,
        #         ARTICLE_CHARS_PER_Q,
        #         EXAMPLE_ARTICLE,
        #         EXAMPLE_ARTICLE_QUESTIONS,
        #     )
        #     for summary, article in zip(summaries, articles)
        # ]
        # print("Generating Coverage QnAs")
        # coverage_qna_sets = await asyncio.gather(*coverage_tasks)

    return {
        "model name": model_name,
        "faithfulness correlations using information density": faithfulness_info_density_correlation_sets,
        "information density mean num questions": information_density_mean_num_questions,
        "faithfulness correlations using fixed number questions": faithfulness_fixed_number_correlation_sets
    }

In [16]:
# summaries = summ_eval["system_output"]
# articles = summ_eval["source"]
# llama3pt2_3b = await qag_eval("llama3.2:3b")

In [ ]:
# Calculate the binned QAG scores - this improves scores and make it more fair against G-Eval
def qag_score_binned(qnas):
    scores = []
    for qna in qnas:
        y_pred = qna["answers"]
        y_true = np.full(y_pred.shape, "yes")
        scores.append(accuracy_score(y_true, y_pred))
        # scores.append(accuracy_score(y_true, y_pred)*100 // 20) # Num Question: 3.0 (0.4802117622174163, 0.39465301511410183, 0.3762859036539601) Num Question: 6.0 (0.5424901170930879, 0.44105068084405996, 0.41546935520534006) Num Question: 9.0 (0.5492987708192336, 0.42572659406116375, 0.39660398370221966)
        # scores.append(np.round(accuracy_score(y_true, y_pred)*100 / 20, 0)) # Num Question: 3.0 (0.48087212140076135, 0.39465301511410183, 0.3762859036539601) Num Question: 6.0  (0.5397768004860264, 0.4409592854101161, 0.41545712509292937), 9.0 (0.5243838456714505, 0.41183323965408347, 0.38919754645723703)

    # partition_size = len(qnas) // 3
    # for i in range(3):
    #     if i == 0:
    #         args = np.argpartition(scores, -partition_size)[-(i+1)*partition_size:]
    #     else:
    #         args = np.argpartition(scores, -partition_size)[-(i+1)*partition_size:-i*partition_size]
        
    #     for arg in args:
    #         scores[arg] = (5 - i) // 1

    return scores


# def qag_score(qnas):
#     return qag_score_binned(qnas)

In [18]:
# summaries = summ_eval["system_output"]
# articles = summ_eval["source"]
# llama3pt1_8b = await qag_eval("llama3.1:8b")

In [20]:
NUM_PARALLEL_CONNS_LLAMA = 1 # mllama doesn't support parallel requests yet
summaries = summ_eval["system_output"][:2]
articles = summ_eval["source"][:2]
llama3pt2_11b = await qag_eval("llama3.2-vision")

Generating Faithfulness QnAs. Num Question: 6.0
	 6.0
	 (1.0, 0.9999999999999999, 1.0)
Generating Faithfulness QnAs. Num Question: 12.0
Not enough questions 11 12
	 11.5
	 (1.0, 0.9999999999999999, 1.0)
Generating Faithfulness QnAs. Num Question: 18.0
Not enough questions 16 18
Not enough questions 12 18
	 14.0
	 (1.0, 0.9999999999999999, 1.0)
Generating Faithfulness QnAs. Num Question: 24.0
Not enough questions 21 24
Not enough questions 19 24
	 20.0
	 (1.0, 0.9999999999999999, 1.0)
Generating Faithfulness QnAs. Num Question: 30.0
Not enough questions 18 30
Not enough questions 21 30
	 19.5
	 (1.0, 0.9999999999999999, 1.0)
Generating Faithfulness QnAs. Num Question: 36.0
Not enough questions 18 36
Not enough questions 20 36
	 19.0
	 (1.0, 0.9999999999999999, 1.0)
Generating Faithfulness QnAs. Num Question: 42.0
Not enough questions 37 42
Not enough questions 27 42
	 32.0
	 (1.0, 0.9999999999999999, 1.0)
Generating Faithfulness QnAs. Num Question: 48.0
Not enough questions 31 48
Not en

CancelledError: 

In [21]:
with open("llama_cache.pickle", "wb") as f:
    pickle.dump(CACHE, f, pickle.HIGHEST_PROTOCOL)

#### Performance

Unsurprisingly, in general the models performed in order of size. The smallest model, Llama 3.2-3B performed the worst, then Llama 3.1-8B, then Llama 3.2-11B, then GPT4o Mini. This result was the same across G-Eval correlation, QAG pearson correlation, QAG spearman correlation, and QAG Kendall-Tau correlation.

Also unsurprisingly, all QAG scores outperformed all G-Eval scores. QAG is a more powerful method capable of 

In [ ]:
model_performances = [llama3pt2_3b, llama3pt1_8b, llama3pt2_11b]
# TODO: plot G-Eval scores
ylims = [[np.inf, -np.inf] for i in range(3)]
for model_performance in model_performances:
    model_name = model_performance["model name"]
    faithfulness_info_density_correlation_sets = model_performance["faithfulness correlations using information density"]
    information_density_mean_num_questions = model_performance["information density mean num questions"]
    faithfulness_fixed_number_correlation_sets = model_performance["faithfulness correlations using fixed number questions"]

    fig, ax = plt.subplots(ncols=3, figsize=(14, 3), num=model_name)
    for i, stat in enumerate(["pearson", "spearman", "kendall-tau"]):
        # Plot the scores generated using information density
        correlation_expert = [correlations[i] for correlations in faithfulness_info_density_correlation_sets]
        ax[i].plot(information_density_mean_num_questions, correlation_expert, label="Information Density")

        # Plot the scores generated using a fixed number of questions
        correlation_expert = [correlations[i] for correlations in faithfulness_fixed_number_correlation_sets]
        ax[i].plot(num_questions_search_grid, correlation_expert, label="Fixed Number of Questions")

        # Plot the G-Eval Score
        g_eval_score = g_eval_correlations[model_name][i]
        ax[i].plot((MIN_QUESTIONS, MAX_QUESTIONS), [g_eval_score]*2, label="G-Eval")
        
        # Set the subplot title and axis labels
        ax[i].set_title(stat)
        ax[i].set_xlabel("Average Number of Questions")
        if i == 0:
            ax[i].set_ylabel("Correlation")

        # Set the x axis limits
        step = num_questions_search_grid[1] - num_questions_search_grid[0]
        ax[i].set_xlim(MIN_QUESTIONS - step, MAX_QUESTIONS + step)

        # Set the y axis limits
        ax_y_lim = ax[i].get_ylim()
        ylims[i][0] = min(ylims[i][0], ax_y_lim[0])
        ylims[i][1] = max(ylims[i][1], ax_y_lim[1])

    # Put the plot's legend in the center of the plot - all subplots use the same legend
    ax[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.25), ncol=1)


for model_performance in model_performances:
    model_name = model_performance["model name"]
    ax = plt.figure(num=model_name).axes
    for i, stat in enumerate(["pearson", "spearman", "kendall-tau"]):
        ax[i].set_ylim(bottom=ylims[i][0], top=ylims[i][1])

plt.show()
plt.close("all")
# # TODO: scatter plot of best performing information density
# plt.scatter(faithfulness_scores, summaries_and_labels["consistency_expert"])
# plt.show()
# plt.close()


In [ ]:
# TODO: get correlation between experts and turkers

#### Citations

[1] Fabbri, Alexander R., et al. "Summeval: Re-evaluating summarization evaluation." Transactions of the Association for Computational Linguistics 9 (2021): 391-409.

[2] Liu, Yang, et al. "G-eval: Nlg evaluation using gpt-4 with better human alignment." arXiv preprint arXiv:2303.16634 (2023).

[3] Maynez, Joshua, et al. "On faithfulness and factuality in abstractive summarization." arXiv preprint arXiv:2005.00661 (2020).

[4] Manakul, Potsawee, Adian Liusie, and Mark JF Gales. "MQAG: Multiple-choice question answering and generation for assessing information consistency in summarization." arXiv preprint arXiv:2301.12307 (2023).

[5] Maneck, B., & Fujimoto, K. (n.d.). Promptflow/examples/flows/evaluation/eval-summarization at main · Microsoft/promptflow. Summarization Evaluation. https://github.com/microsoft/promptflow/tree/main/examples/flows/evaluation/eval-summarization#meta-evaluation 

[6] Confident AI Inc. (2024). The Open-Source LLM Evaluation Framework. DeepEval. https://docs.confident-ai.com/ 

[7] TruEra. (2024). TruLens for LLMs. TruLens. https://www.trulens.org/ 